# Blue Catalyst FG Batch Runbook

Stages:
- plan: build frozen embedding index + batch manifests
- merge: reconcile FG batch outputs with frozen embeddings


In [ ]:
import os
import subprocess
from pathlib import Path

root = Path(os.getenv('METHANET_ROOT', '.')).resolve()
script = root / 'scripts' / 'blue_catalyst_fg_batch_pipeline.py'
stage = os.getenv('BC_FG_STAGE', 'plan').strip().lower()
source_run = os.getenv('BC_FG_SOURCE_EMBED_RUN_ID', 'unknown_embed')
art_default = root / 'results' / 'blue_catalyst_poc' / 'runs'
art_default = art_default / 'fg_runbook' / 'fg_artifacts'
artifacts = Path(os.getenv('BC_FG_ARTIFACTS_DIR', str(art_default)))
artifacts = artifacts.expanduser().resolve()
batch_size = int(os.getenv('BC_FG_BATCH_SIZE', '25'))
min_join = float(os.getenv('BC_FG_MIN_JOIN_COVERAGE', '0.95'))
hash_prot = os.getenv('BC_FG_HASH_PROTEOMES', '0') == '1'
embed_meta = os.getenv('BC_FG_EMBED_METADATA', '').strip()
embed_npz = os.getenv('BC_FG_EMBED_NPZ', '').strip()
artifacts.mkdir(parents=True, exist_ok=True)

if stage == 'plan':
    if not embed_meta or not embed_npz:
        raise RuntimeError('Missing BC_FG_EMBED_METADATA/BC_FG_EMBED_NPZ')
    cmd = ['python', str(script), 'plan']
    cmd += ['--embedding-metadata', embed_meta]
    cmd += ['--embedding-npz', embed_npz]
    cmd += ['--embedding-run-id', source_run]
    cmd += ['--output-dir', str(artifacts)]
    cmd += ['--batch-size', str(batch_size)]
    if hash_prot:
        cmd.append('--hash-proteomes')
elif stage == 'merge':
    cmd = ['python', str(script), 'merge']
    cmd += ['--fg-plan-dir', str(artifacts)]
    cmd += ['--batch-results-dir', str(artifacts / 'batch_results')]
    cmd += ['--output-dir', str(artifacts)]
    cmd += ['--min-join-coverage', str(min_join)]
else:
    raise RuntimeError(f'Unsupported BC_FG_STAGE={stage}')

print('Executing:', ' '.join(cmd))
subprocess.run(cmd, check=True)
